In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pickle
import os

# Load dataset
print('Loading AI4I 2020 dataset...')
df = pd.read_csv('data/ai4i2020.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nColumn names: {list(df.columns)}')
print(f'\nFirst few rows:')
print(df.head())
print(f'\nDataset info:')
print(df.info())
print(f'\nMissing values:')
print(df.isnull().sum())
print(f'\nMachine failure distribution:')
print(df['Machine failure'].value_counts())

In [ ]:
# Data preprocessing
print('\nPreprocessing data...')

# Drop UDI and Product ID (identifiers, not features)
X = df.drop(['UDI', 'Product ID', 'Machine failure'], axis=1)
y = df['Machine failure']

# Encode Type column
le_type = LabelEncoder()
X['Type'] = le_type.fit_transform(X['Type'])

print(f'Feature matrix shape: {X.shape}')
print(f'Target distribution: {y.value_counts().to_dict()}')
print(f'Failure rate: {y.sum() / len(y) * 100:.2f}%')

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'\nTrain set size: {X_train.shape[0]}')
print(f'Test set size: {X_test.shape[0]}')
print(f'Train failure rate: {y_train.sum() / len(y_train) * 100:.2f}%')
print(f'Test failure rate: {y_test.sum() / len(y_test) * 100:.2f}%')

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('\nData preprocessing complete')

In [ ]:
# Train Gradient Boosting model
print('\nTraining Gradient Boosting Classifier...')
gb_model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    verbose=0
)
gb_model.fit(X_train_scaled, y_train)

# Predictions
y_pred_gb = gb_model.predict(X_test_scaled)
y_pred_proba_gb = gb_model.predict_proba(X_test_scaled)[:, 1]

# Evaluation
print('\nGradient Boosting Results:')
print(f'ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba_gb):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_gb, target_names=['No Failure', 'Failure']))
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_gb))

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': gb_model.feature_importances_
}).sort_values('importance', ascending=False)

print('\nTop Feature Importances:')
print(feature_importance.head(10))

In [ ]:
# Save model and preprocessing objects
print('\nSaving model and preprocessing objects...')

model_dir = 'src/models'
os.makedirs(model_dir, exist_ok=True)

# Save model
with open(f'{model_dir}/failure_predictor.pkl', 'wb') as f:
    pickle.dump(gb_model, f)
print(f'✓ Model saved to {model_dir}/failure_predictor.pkl')

# Save scaler
with open(f'{model_dir}/feature_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f'✓ Scaler saved to {model_dir}/feature_scaler.pkl')

# Save label encoder
with open(f'{model_dir}/type_encoder.pkl', 'wb') as f:
    pickle.dump(le_type, f)
print(f'✓ Type encoder saved to {model_dir}/type_encoder.pkl')

# Save feature names
feature_names = list(X.columns)
with open(f'{model_dir}/feature_names.txt', 'w') as f:
    f.write('\n'.join(feature_names))
print(f'✓ Feature names saved to {model_dir}/feature_names.txt')

print('\n✓ All artifacts saved successfully')

In [ ]:
# Create prediction helper function
import json

def predict_machine_failure(features_dict):
    """
    Predict machine failure probability from operational features.
    
    Args:
        features_dict: Dict with keys like 'Type', 'Air temperature [K]', etc.
    
    Returns:
        Dict with 'failure_risk', 'failure_probability', and 'confidence'
    """
    # Load model and scaler
    with open('src/models/failure_predictor.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('src/models/feature_scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    with open('src/models/type_encoder.pkl', 'rb') as f:
        type_enc = pickle.load(f)
    
    # Extract features in correct order
    feature_cols = ['Type', 'Air temperature [K]', 'Process temperature [K]', 
                   'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
    
    # Prepare feature vector
    feature_values = []
    for col in feature_cols:
        if col == 'Type':
            val = type_enc.transform([features_dict.get(col, 'M')])[0]
        else:
            val = float(features_dict.get(col, 0))
        feature_values.append(val)
    
    X_input = np.array([feature_values]).reshape(1, -1)
    X_scaled = scaler.transform(X_input)
    
    # Predict
    prob = model.predict_proba(X_scaled)[0, 1]
    pred = model.predict(X_scaled)[0]
    
    # Map to risk level
    if prob < 0.3:
        risk_level = 'Low'
    elif prob < 0.6:
        risk_level = 'Medium'
    elif prob < 0.8:
        risk_level = 'High'
    else:
        risk_level = 'Critical'
    
    return {
        'failure_risk': risk_level,
        'failure_probability': float(prob),
        'predicted_failure': int(pred),
        'confidence': float(max(model.predict_proba(X_scaled)[0]))
    }

# Save prediction function
with open('src/models/predict_failure.py', 'w') as f:
    f.write("""import pickle
import numpy as np

def predict_machine_failure(features_dict):
    with open('src/models/failure_predictor.pkl', 'rb') as f:
        model = pickle.load(f)
    with open('src/models/feature_scaler.pkl', 'rb') as f:
        scaler = pickle.load(f)
    with open('src/models/type_encoder.pkl', 'rb') as f:
        type_enc = pickle.load(f)
    
    feature_cols = ['Type', 'Air temperature [K]', 'Process temperature [K]', 
                   'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
    
    feature_values = []
    for col in feature_cols:
        if col == 'Type':
            val = type_enc.transform([features_dict.get(col, 'M')])[0]
        else:
            val = float(features_dict.get(col, 0))
        feature_values.append(val)
    
    X_input = np.array([feature_values]).reshape(1, -1)
    X_scaled = scaler.transform(X_input)
    
    prob = model.predict_proba(X_scaled)[0, 1]
    pred = model.predict(X_scaled)[0]
    
    if prob < 0.3:
        risk_level = 'Low'
    elif prob < 0.6:
        risk_level = 'Medium'
    elif prob < 0.8:
        risk_level = 'High'
    else:
        risk_level = 'Critical'
    
    return {
        'failure_risk': risk_level,
        'failure_probability': float(prob),
        'predicted_failure': int(pred),
        'confidence': float(max(model.predict_proba(X_scaled)[0]))
    }
""")

print('✓ Prediction function saved')

In [ ]:
# Test the prediction function with sample data
test_sample = {
    'Type': 'M',
    'Air temperature [K]': 298.1,
    'Process temperature [K]': 308.6,
    'Rotational speed [rpm]': 1551,
    'Torque [Nm]': 42.8,
    'Tool wear [min]': 0
}

prediction = predict_machine_failure(test_sample)
print('\nSample Prediction:')
print(f'Input: {test_sample}')
print(f'Output: {prediction}')